# Fine-tune XTTS-v2 on Haryanvi TTS

This notebook fine-tunes XTTS-v2 for a college Hindi/Haryanvi TTS project using the Hugging Face dataset [`ankitdhiman/haryanvi-tts`](https://huggingface.co/datasets/ankitdhiman/haryanvi-tts).

The dataset is a Hugging Face `soundfolder` dataset with audio/text pairs in the Bangru dialect of Haryanvi. The dataset card reports Apache-2.0 licensing, about 5.5k train rows, and a `metadata.csv` file mapping file paths to text.

Recommended flow:

1. Run once with `SMOKE_TEST=True` to verify preprocessing, training startup, and export.
2. Switch `SMOKE_TEST=False` for the real A100 run.
3. In Colab, keep `USE_GOOGLE_DRIVE=True`; checkpoints and export files are written to Drive for resume after runtime resets.
4. Upload `hf_export` to Hugging Face.
5. Set this app's XTTS env vars to the uploaded repo.

Outputs are prepared for this app's Docker/Modal deployment:

```bash
TTS_BACKEND=xtts
XTTS_LOADER=native
XTTS_REPO_ID=your-org/haryanvi-xtts-v2
XTTS_CHECKPOINT_FILENAME=model.pth
XTTS_CONFIG_FILENAME=config.json
XTTS_VOCAB_FILENAME=vocab.json
XTTS_DVAE_FILENAME=dvae.pth
XTTS_MEL_STATS_FILENAME=mel_stats.pth
XTTS_SPEAKERS_FILENAME=speakers_xtts.pth
XTTS_SPEAKER_WAV_FILENAME=reference.wav
XTTS_LANGUAGE=hi
COQUI_TOS_AGREED=1
```

Review XTTS-v2 license terms before using `COQUI_TOS_AGREED=1`.

In [ ]:
# Run once per fresh Colab/runtime.
!pip install -q --upgrade pip setuptools wheel
!pip install -q -U coqui-tts coqui-tts-trainer datasets huggingface_hub soundfile librosa pandas scikit-learn torch torchaudio


In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import json
import os
import random
import shutil

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# Required by some XTTS-v2 downloads/inference paths. Set only after reviewing terms.
os.environ.setdefault("COQUI_TOS_AGREED", "1")


def _bool_env(name: str, default: str = "false") -> bool:
    return os.getenv(name, default).lower() in {"1", "true", "yes", "y", "on"}


IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    drive = None

USE_GOOGLE_DRIVE_DEFAULT = _bool_env("USE_GOOGLE_DRIVE", "true") and IN_COLAB
DRIVE_ROOT_DEFAULT = Path(os.getenv("XTTS_DRIVE_ROOT", "/content/drive/MyDrive/haryanvi_xtts_v2"))
LOCAL_ROOT_DEFAULT = Path(os.getenv("XTTS_LOCAL_ROOT", "xtts_haryanvi_outputs"))
OUTPUT_ROOT_DEFAULT = DRIVE_ROOT_DEFAULT if USE_GOOGLE_DRIVE_DEFAULT else LOCAL_ROOT_DEFAULT

if USE_GOOGLE_DRIVE_DEFAULT and drive is not None:
    drive.mount("/content/drive")


@dataclass
class XTTSFineTuneConfig:
    # Data
    DATASET_ID: str = "ankitdhiman/haryanvi-tts"
    DATASET_SPLIT: str = "train"
    TEXT_COLUMN: Optional[str] = None  # auto-detect from common names
    AUDIO_COLUMN: str = "audio"
    MAX_SAMPLES: Optional[int] = None
    EVAL_SIZE: float = 0.05
    RANDOM_SEED: int = 42

    # Fast validation run before spending A100 time.
    SMOKE_TEST: bool = False
    SMOKE_TEST_SAMPLES: int = 100

    # Drive/resume support. In Colab, checkpoints and final export go to Drive.
    USE_GOOGLE_DRIVE: bool = USE_GOOGLE_DRIVE_DEFAULT
    AUTO_RESUME_LATEST: bool = _bool_env("XTTS_AUTO_RESUME_LATEST", "false")

    # Audio hygiene. XTTS GPT fine-tuning works best with clean, sentence-length clips.
    MIN_AUDIO_SECONDS: float = 2.0
    MAX_AUDIO_SECONDS: float = 12.0
    REFERENCE_MIN_SECONDS: float = 8.0
    REFERENCE_MAX_SECONDS: float = 13.0

    # Paths. Dataset/base downloads stay local; training checkpoints/export go to Drive in Colab.
    PROJECT_ROOT: Path = OUTPUT_ROOT_DEFAULT
    LOCAL_WORK_DIR: Path = LOCAL_ROOT_DEFAULT
    COQUI_DATASET_DIR: Path = LOCAL_ROOT_DEFAULT / "coqui_dataset"
    XTTS_BASE_DIR: Path = LOCAL_ROOT_DEFAULT / "xtts_base"
    TRAINING_OUTPUT_DIR: Path = OUTPUT_ROOT_DEFAULT / "training"
    EXPORT_DIR: Path = OUTPUT_ROOT_DEFAULT / "hf_export"

    # Base XTTS-v2 artifacts
    XTTS_BASE_REPO_ID: str = "coqui/XTTS-v2"
    XTTS_BASE_CHECKPOINT: str = "model.pth"
    XTTS_BASE_CONFIG: str = "config.json"
    XTTS_BASE_VOCAB: str = "vocab.json"
    XTTS_BASE_DVAE: str = "dvae.pth"
    XTTS_BASE_MEL_STATS: str = "mel_stats.pth"
    XTTS_BASE_SPEAKERS: str = "speakers_xtts.pth"

    # A100 defaults. If you hit OOM, use BATCH_SIZE=4 and GRAD_ACCUM_STEPS=4.
    RUN_NAME: str = "haryanvi_xtts_v2_gpt"
    LANGUAGE: str = "hi"
    NUM_EPOCHS: int = 6
    BATCH_SIZE: int = 8
    GRAD_ACCUM_STEPS: int = 2
    EVAL_BATCH_SIZE: int = 8
    NUM_WORKERS: int = 4
    LR: float = 5e-6
    SAVE_STEP: int = 500
    PRINT_STEP: int = 25
    MAX_TEXT_LENGTH: int = 200
    MAX_WAV_LENGTH: int = 255995  # around 11.6s at 22050 Hz
    RESUME_PATH: Optional[str] = os.getenv("XTTS_RESUME_PATH")  # set to a Drive checkpoint path to resume

    # Manual HF upload target after training
    HF_EXPORT_REPO_ID: str = os.getenv("HF_EXPORT_REPO_ID", "your-username/haryanvi-xtts-v2")


CFG = XTTSFineTuneConfig()
if CFG.SMOKE_TEST:
    CFG.MAX_SAMPLES = CFG.SMOKE_TEST_SAMPLES
    CFG.NUM_EPOCHS = 1
    CFG.BATCH_SIZE = min(CFG.BATCH_SIZE, 4)
    CFG.EVAL_BATCH_SIZE = min(CFG.EVAL_BATCH_SIZE, 4)
    CFG.GRAD_ACCUM_STEPS = 2

for path in [
    CFG.PROJECT_ROOT,
    CFG.LOCAL_WORK_DIR,
    CFG.COQUI_DATASET_DIR,
    CFG.XTTS_BASE_DIR,
    CFG.TRAINING_OUTPUT_DIR,
    CFG.EXPORT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

if CFG.AUTO_RESUME_LATEST and CFG.RESUME_PATH is None:
    resume_candidates = []
    for pattern in ["best_model.pth", "model.pth", "checkpoint_*.pth"]:
        resume_candidates.extend(CFG.TRAINING_OUTPUT_DIR.rglob(pattern))
    if resume_candidates:
        latest_checkpoint = max(resume_candidates, key=lambda p: p.stat().st_mtime)
        CFG.RESUME_PATH = str(latest_checkpoint)
        print("Auto resume checkpoint:", CFG.RESUME_PATH)

random.seed(CFG.RANDOM_SEED)
print(CFG)
print("Training checkpoints directory:", CFG.TRAINING_OUTPUT_DIR)
print("Export directory:", CFG.EXPORT_DIR)

In [ ]:
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from datasets import Audio, load_dataset
from huggingface_hub import snapshot_download
from sklearn.model_selection import train_test_split

try:
    from google.colab import userdata
    if not os.getenv("HF_TOKEN"):
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
except Exception:
    pass

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


In [ ]:
print("Loading dataset...")
ds = load_dataset(CFG.DATASET_ID, split=CFG.DATASET_SPLIT)
print(ds)
print("Columns:", ds.column_names)

if CFG.MAX_SAMPLES:
    ds = ds.shuffle(seed=CFG.RANDOM_SEED).select(range(min(CFG.MAX_SAMPLES, len(ds))))

if CFG.AUDIO_COLUMN not in ds.column_names:
    raise ValueError(f"Expected audio column {CFG.AUDIO_COLUMN!r}. Found: {ds.column_names}")

text_candidates = [
    CFG.TEXT_COLUMN,
    "text",
    "sentence",
    "transcription",
    "normalized_text",
    "haryanvi",
    "metadata",
]
text_col = next((col for col in text_candidates if col and col in ds.column_names), None)
if text_col is None:
    raise ValueError(
        "Could not auto-detect the text column. "
        f"Found columns: {ds.column_names}. Set CFG.TEXT_COLUMN manually and rerun."
    )

print("Using text column:", text_col)

ds = ds.cast_column(CFG.AUDIO_COLUMN, Audio(sampling_rate=22050))
print(ds[0])


In [ ]:
# Export Hugging Face soundfolder rows into Coqui/LJSpeech-style metadata.
# File format: wavs/<id>.wav|text|text
import re

wav_dir = CFG.COQUI_DATASET_DIR / "wavs"
if wav_dir.exists():
    shutil.rmtree(wav_dir)
wav_dir.mkdir(parents=True, exist_ok=True)

rows = []
skipped_duration = 0
skipped_empty_text = 0
for idx, item in enumerate(ds):
    text = str(item[text_col]).strip()
    text = re.sub(r"\s+", " ", text)
    if not text:
        skipped_empty_text += 1
        continue

    audio = item[CFG.AUDIO_COLUMN]
    array = np.asarray(audio["array"], dtype=np.float32)
    sr = int(audio["sampling_rate"])
    if array.ndim > 1:
        array = np.mean(array, axis=1)

    duration_seconds = len(array) / float(sr)
    if not (CFG.MIN_AUDIO_SECONDS <= duration_seconds <= CFG.MAX_AUDIO_SECONDS):
        skipped_duration += 1
        continue

    filename = f"utt_{idx:06d}.wav"
    rel_path = f"wavs/{filename}"
    out_path = wav_dir / filename
    sf.write(out_path, array, sr)
    rows.append({
        "audio_file": rel_path,
        "text": text,
        "duration_seconds": duration_seconds,
    })

metadata_df = pd.DataFrame(rows).drop_duplicates(subset=["audio_file", "text"]).reset_index(drop=True)
if metadata_df.empty:
    raise ValueError("No usable audio rows after filtering. Relax MIN_AUDIO_SECONDS/MAX_AUDIO_SECONDS.")

train_df, eval_df = train_test_split(
    metadata_df,
    test_size=CFG.EVAL_SIZE,
    random_state=CFG.RANDOM_SEED,
)
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

def write_ljspeech_metadata(frame: pd.DataFrame, path: Path) -> None:
    with path.open("w", encoding="utf-8") as f:
        for row in frame.itertuples(index=False):
            f.write(f"{row.audio_file}|{row.text}|{row.text}\n")

write_ljspeech_metadata(train_df, CFG.COQUI_DATASET_DIR / "metadata_train.csv")
write_ljspeech_metadata(eval_df, CFG.COQUI_DATASET_DIR / "metadata_eval.csv")

reference_pool = train_df[
    train_df["duration_seconds"].between(CFG.REFERENCE_MIN_SECONDS, CFG.REFERENCE_MAX_SECONDS)
]
if reference_pool.empty:
    reference_pool = train_df
reference_idx = (reference_pool["duration_seconds"] - 10.0).abs().idxmin()
reference_row = reference_pool.loc[reference_idx]
reference_wav = CFG.EXPORT_DIR / "reference.wav"
reference_source = CFG.COQUI_DATASET_DIR / reference_row["audio_file"]
shutil.copy2(reference_source, reference_wav)

print("Prepared Coqui dataset:", CFG.COQUI_DATASET_DIR)
print("Train rows:", len(train_df), "Eval rows:", len(eval_df))
print("Skipped empty text:", skipped_empty_text)
print("Skipped by duration:", skipped_duration)
print("Duration stats:")
print(metadata_df["duration_seconds"].describe())
print("Reference wav:", reference_wav)
print("Reference duration:", round(float(reference_row["duration_seconds"]), 2), "seconds")
print("Reference text:", reference_row["text"])
print(train_df.head())

In [ ]:
# Download XTTS-v2 base files.
# If this is interrupted, rerun; snapshot_download resumes from cache.
base_snapshot = snapshot_download(
    repo_id=CFG.XTTS_BASE_REPO_ID,
    local_dir=str(CFG.XTTS_BASE_DIR),
    local_dir_use_symlinks=False,
    allow_patterns=[
        CFG.XTTS_BASE_CHECKPOINT,
        CFG.XTTS_BASE_CONFIG,
        CFG.XTTS_BASE_VOCAB,
        CFG.XTTS_BASE_DVAE,
        CFG.XTTS_BASE_MEL_STATS,
        CFG.XTTS_BASE_SPEAKERS,
    ],
)
print("XTTS base snapshot:", base_snapshot)
print(sorted(p.name for p in CFG.XTTS_BASE_DIR.iterdir()))


In [ ]:
# Build the Coqui XTTS GPT fine-tuning trainer config.
from trainer import Trainer, TrainerArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import (
    GPTArgs,
    GPTTrainer,
    GPTTrainerConfig,
    XttsAudioConfig,
)

xtts_checkpoint = CFG.XTTS_BASE_DIR / CFG.XTTS_BASE_CHECKPOINT
dvae_checkpoint = CFG.XTTS_BASE_DIR / CFG.XTTS_BASE_DVAE
mel_norm_file = CFG.XTTS_BASE_DIR / CFG.XTTS_BASE_MEL_STATS
tokenizer_file = CFG.XTTS_BASE_DIR / CFG.XTTS_BASE_VOCAB

for required in [xtts_checkpoint, dvae_checkpoint, mel_norm_file, tokenizer_file]:
    if not required.exists():
        raise FileNotFoundError(required)

dataset_config = BaseDatasetConfig(
    formatter="ljspeech",
    meta_file_train="metadata_train.csv",
    meta_file_val="metadata_eval.csv",
    path=str(CFG.COQUI_DATASET_DIR),
    language=CFG.LANGUAGE,
)

model_args = GPTArgs(
    max_conditioning_length=132300,
    min_conditioning_length=66150,
    debug_loading_failures=False,
    max_wav_length=CFG.MAX_WAV_LENGTH,
    max_text_length=CFG.MAX_TEXT_LENGTH,
    mel_norm_file=str(mel_norm_file),
    dvae_checkpoint=str(dvae_checkpoint),
    xtts_checkpoint=str(xtts_checkpoint),
    tokenizer_file=str(tokenizer_file),
    gpt_num_audio_tokens=1026,
    gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True,
    gpt_use_perceiver_resampler=True,
)

audio_config = XttsAudioConfig(
    sample_rate=22050,
    dvae_sample_rate=22050,
    output_sample_rate=24000,
)

config = GPTTrainerConfig(
    output_path=str(CFG.TRAINING_OUTPUT_DIR),
    model_args=model_args,
    run_name=CFG.RUN_NAME,
    project_name="haryanvi_xtts_v2",
    run_description="Fine-tune XTTS-v2 GPT encoder on Haryanvi/Bangru speech",
    dashboard_logger="tensorboard",
    logger_uri=None,
    audio=audio_config,
    batch_size=CFG.BATCH_SIZE,
    batch_group_size=48,
    eval_batch_size=CFG.EVAL_BATCH_SIZE,
    num_loader_workers=CFG.NUM_WORKERS,
    eval_split_max_size=256,
    print_step=CFG.PRINT_STEP,
    plot_step=100,
    log_model_step=CFG.SAVE_STEP,
    save_step=CFG.SAVE_STEP,
    save_n_checkpoints=2,
    save_checkpoints=True,
    print_eval=False,
    optimizer="AdamW",
    optimizer_wd_only_on_weights=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=CFG.LR,
    lr_scheduler="MultiStepLR",
    lr_scheduler_params={"milestones": [50000, 100000, 200000], "gamma": 0.5},
    epochs=CFG.NUM_EPOCHS,
    test_sentences=[
        {
            "text": "राम राम, आज मौसम घणा बढ़िया सै।",
            "speaker_wav": str(CFG.EXPORT_DIR / "reference.wav"),
            "language": CFG.LANGUAGE,
        },
        {
            "text": "म्हने बेरा कोन्या के ओ कड़े ग्या।",
            "speaker_wav": str(CFG.EXPORT_DIR / "reference.wav"),
            "language": CFG.LANGUAGE,
        },
    ],
)

config_path = CFG.TRAINING_OUTPUT_DIR / "xtts_gpt_trainer_config.json"
config.save_json(str(config_path))
print("Saved trainer config:", config_path)


In [ ]:
# Load samples and train.
# For a quick smoke test, set CFG.MAX_SAMPLES=100 and CFG.NUM_EPOCHS=1 above.
train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=False,
)

# Because we already created explicit train/eval metadata files, load eval separately.
eval_dataset_config = BaseDatasetConfig(
    formatter="ljspeech",
    meta_file_train="metadata_eval.csv",
    path=str(CFG.COQUI_DATASET_DIR),
    language=CFG.LANGUAGE,
)
eval_samples, _ = load_tts_samples(eval_dataset_config, eval_split=False)

print("Train samples:", len(train_samples))
print("Eval samples:", len(eval_samples))

model = GPTTrainer.init_from_config(config)
trainer = Trainer(
    TrainerArgs(
        restore_path=CFG.RESUME_PATH,
        skip_train_epoch=False,
        start_with_eval=True,
        grad_accum_steps=CFG.GRAD_ACCUM_STEPS,
    ),
    config,
    output_path=str(CFG.TRAINING_OUTPUT_DIR),
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)
trainer.fit()

print("Training checkpoints directory:", CFG.TRAINING_OUTPUT_DIR)
print("If Colab disconnects, set XTTS_RESUME_PATH to a checkpoint in this directory and rerun from the config cell.")

In [ ]:
# Export files in the exact form this app expects from Hugging Face.
# The app uses:
#   XTTS_LOADER=native
#   XTTS_CHECKPOINT_FILENAME=model.pth
#   XTTS_CONFIG_FILENAME=config.json
#   XTTS_VOCAB_FILENAME=vocab.json
#   XTTS_DVAE_FILENAME=dvae.pth
#   XTTS_MEL_STATS_FILENAME=mel_stats.pth
#   XTTS_SPEAKERS_FILENAME=speakers_xtts.pth
#   XTTS_SPEAKER_WAV_FILENAME=reference.wav

candidate_names = ["best_model.pth", "model.pth"]
checkpoint_candidates = []
for name in candidate_names:
    checkpoint_candidates.extend(CFG.TRAINING_OUTPUT_DIR.rglob(name))
checkpoint_candidates.extend(CFG.TRAINING_OUTPUT_DIR.rglob("checkpoint_*.pth"))
checkpoint_candidates = sorted(set(checkpoint_candidates), key=lambda p: p.stat().st_mtime, reverse=True)

if not checkpoint_candidates:
    raise FileNotFoundError(f"No trained XTTS checkpoint found under {CFG.TRAINING_OUTPUT_DIR}")

best_checkpoint = checkpoint_candidates[0]
print("Selected checkpoint:", best_checkpoint)

CFG.EXPORT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_checkpoint, CFG.EXPORT_DIR / "model.pth")
# Use the XTTS inference config, not the trainer config, for app deployment.
shutil.copy2(CFG.XTTS_BASE_DIR / CFG.XTTS_BASE_CONFIG, CFG.EXPORT_DIR / "config.json")
shutil.copy2(CFG.XTTS_BASE_DIR / CFG.XTTS_BASE_VOCAB, CFG.EXPORT_DIR / "vocab.json")
shutil.copy2(CFG.XTTS_BASE_DIR / CFG.XTTS_BASE_DVAE, CFG.EXPORT_DIR / "dvae.pth")
shutil.copy2(CFG.XTTS_BASE_DIR / CFG.XTTS_BASE_MEL_STATS, CFG.EXPORT_DIR / "mel_stats.pth")

base_speakers = CFG.XTTS_BASE_DIR / CFG.XTTS_BASE_SPEAKERS
if base_speakers.exists():
    shutil.copy2(base_speakers, CFG.EXPORT_DIR / "speakers_xtts.pth")

if not (CFG.EXPORT_DIR / "reference.wav").exists():
    shutil.copy2(reference_wav, CFG.EXPORT_DIR / "reference.wav")

required_export_files = [
    "model.pth",
    "config.json",
    "vocab.json",
    "dvae.pth",
    "mel_stats.pth",
    "reference.wav",
]
missing = [name for name in required_export_files if not (CFG.EXPORT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Export missing required files: {missing}")

# Validate that the exported config is loadable by the same native XTTS path used by the app.
from TTS.tts.configs.xtts_config import XttsConfig
_export_config = XttsConfig()
_export_config.load_json(str(CFG.EXPORT_DIR / "config.json"))

manifest = {
    "base_model": CFG.XTTS_BASE_REPO_ID,
    "dataset": CFG.DATASET_ID,
    "language": CFG.LANGUAGE,
    "checkpoint_source": str(best_checkpoint),
    "reference_wav": "reference.wav",
    "reference_duration_seconds": float(reference_row["duration_seconds"]),
    "training": {
        "num_epochs": CFG.NUM_EPOCHS,
        "batch_size": CFG.BATCH_SIZE,
        "grad_accum_steps": CFG.GRAD_ACCUM_STEPS,
        "effective_batch_size": CFG.BATCH_SIZE * CFG.GRAD_ACCUM_STEPS,
        "learning_rate": CFG.LR,
        "max_samples": CFG.MAX_SAMPLES,
    },
    "audio_filter": {
        "min_audio_seconds": CFG.MIN_AUDIO_SECONDS,
        "max_audio_seconds": CFG.MAX_AUDIO_SECONDS,
    },
    "app_env": {
        "TTS_BACKEND": "xtts",
        "XTTS_LOADER": "native",
        "XTTS_REPO_ID": CFG.HF_EXPORT_REPO_ID,
        "XTTS_CHECKPOINT_FILENAME": "model.pth",
        "XTTS_CONFIG_FILENAME": "config.json",
        "XTTS_VOCAB_FILENAME": "vocab.json",
        "XTTS_DVAE_FILENAME": "dvae.pth",
        "XTTS_MEL_STATS_FILENAME": "mel_stats.pth",
        "XTTS_SPEAKERS_FILENAME": "speakers_xtts.pth",
        "XTTS_SPEAKER_WAV_FILENAME": "reference.wav",
        "XTTS_LANGUAGE": CFG.LANGUAGE,
        "COQUI_TOS_AGREED": "1",
    },
}
with open(CFG.EXPORT_DIR / "haryanvi_xtts_export_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

readme = f"""---
language:
- hi
tags:
- text-to-speech
- xtts
- haryanvi
- bangru
license: other
---

# Haryanvi XTTS-v2 Fine-tune

Base model: `{CFG.XTTS_BASE_REPO_ID}`

Dataset: `{CFG.DATASET_ID}`

This repo is exported for native XTTS-v2 loading in the Hindi/Haryanvi demo app.
Review the base XTTS-v2 license/terms before public or commercial use.

## App env

```bash
TTS_BACKEND=xtts
XTTS_LOADER=native
XTTS_REPO_ID={CFG.HF_EXPORT_REPO_ID}
XTTS_CHECKPOINT_FILENAME=model.pth
XTTS_CONFIG_FILENAME=config.json
XTTS_VOCAB_FILENAME=vocab.json
XTTS_DVAE_FILENAME=dvae.pth
XTTS_MEL_STATS_FILENAME=mel_stats.pth
XTTS_SPEAKERS_FILENAME=speakers_xtts.pth
XTTS_SPEAKER_WAV_FILENAME=reference.wav
XTTS_LANGUAGE={CFG.LANGUAGE}
COQUI_TOS_AGREED=1
```
"""
(CFG.EXPORT_DIR / "README.md").write_text(readme, encoding="utf-8")

print("Export directory:", CFG.EXPORT_DIR)
print(sorted(p.name for p in CFG.EXPORT_DIR.iterdir()))
print("Manual upload command:")
print(f"huggingface-cli upload {CFG.HF_EXPORT_REPO_ID} {CFG.EXPORT_DIR}")

In [ ]:
# Optional: upload from the notebook instead of manual CLI upload.
# Uncomment after setting CFG.HF_EXPORT_REPO_ID and HF_TOKEN.

# from huggingface_hub import HfApi
# api = HfApi(token=os.getenv("HF_TOKEN"))
# api.create_repo(CFG.HF_EXPORT_REPO_ID, repo_type="model", private=True, exist_ok=True)
# api.upload_folder(
#     folder_path=str(CFG.EXPORT_DIR),
#     repo_id=CFG.HF_EXPORT_REPO_ID,
#     repo_type="model",
#     commit_message="Upload fine-tuned Haryanvi XTTS-v2",
# )
# print(f"Uploaded: https://huggingface.co/{CFG.HF_EXPORT_REPO_ID}")


In [ ]:
# Quick app env block after upload.
print(f"""
TTS_BACKEND=xtts
XTTS_LOADER=native
XTTS_REPO_ID={CFG.HF_EXPORT_REPO_ID}
XTTS_CHECKPOINT_FILENAME=model.pth
XTTS_CONFIG_FILENAME=config.json
XTTS_VOCAB_FILENAME=vocab.json
XTTS_DVAE_FILENAME=dvae.pth
XTTS_MEL_STATS_FILENAME=mel_stats.pth
XTTS_SPEAKERS_FILENAME=speakers_xtts.pth
XTTS_SPEAKER_WAV_FILENAME=reference.wav
XTTS_LANGUAGE={CFG.LANGUAGE}
COQUI_TOS_AGREED=1
""")